# **Thông tin nhóm**
- Lớp: ML 23KHDL1
- Nhóm: 6
- Sinh viên:
    - 23127102 - Lê Quang Phúc
    - 23127212 - Nguyễn Quang Đăng Khoa
    - 23127241 - Đoàn Thành Phát
    - 23127332 - Trần Tiến Cường
    - 23127442 - Trầm Hữu Nhân


# **Đánh giá baseline đối với mô hình GLMOCR**
Mục tiêu của notebook này là đánh giá hiệu quả của mô hình `GLMOCR` trong bài toán nhận diện ký tự quang học (OCR) trên bộ dữ liệu đã chuẩn bị. Kết quả đánh giá sẽ giúp so sánh chất lượng giữa các mô hình OCR khác nhau.

## 1. Cài đặt thư viện và môi trường

In [ ]:
!pip install -U transformers accelerate tiktoken
!pip install -q Levenshtein editdistance pandas tqdm

### 1.1 Mount tới Google Drive để lấy dataset

Do tập dữ liệu có kích thước lớn, việc đọc trực tiếp từ **Google Drive** có thể gây ra hiện tượng thắt nút cổ chai băng thông làm chậm đáng kể quá trình suy luận của mô hình. Nên là:
- **Mount Google Drive** để lấy file nén `processed_data.zip`.
- **Giải nén trực tiếp vào bộ nhớ cục bộ** của máy ảo Colab (`/content/local_data`). Thao tác này giúp thao tác đọc ảnh trong vòng lặp đánh giá sau này đạt tốc độ tối đa.

In [ ]:
import os
import json
import unicodedata
import pandas as pd
import Levenshtein
import torch
import importlib

from pathlib import Path
from tqdm import tqdm
from PIL import Image
from transformers import AutoProcessor, AutoConfig
from google.colab import drive

# Kết nối Google Drive
drive.mount('/content/drive')

# Đường dẫn
ZIP_PATH = Path('/content/drive/MyDrive/IntroToML - OCR - data/processed_data.zip')
LOCAL_ROOT = Path('/content/local_data')

# Giải nén
if ZIP_PATH.exists():
    if not LOCAL_ROOT.exists():
        !unzip -q "{ZIP_PATH}" -d "{LOCAL_ROOT}"
    else:
        print("Dữ liệu đã có sẵn.")
else:
    print(f"❌ LỖI: Không tìm thấy file {ZIP_PATH}")

TEST_DIR = LOCAL_ROOT / 'test'

print(f"\nĐường dẫn thư mục Test: {TEST_DIR}")

Mounted at /content/drive

Đường dẫn thư mục Test: /content/local_data/test


## 2. Các hàm hỗ trợ
Trước khi đưa dữ liệu vào mô hình, ta cần chuẩn hóa văn bản. Trong tiếng Việt, hiện tượng khác biệt bảng mã `Unicode` rất phổ biến và có thể dẫn đến việc đánh giá sai lệch dù mặt chữ giống nhau. 
Hàm `normalize_text` sử dụng chuẩn **NFC** để đưa toàn bộ văn bản về một dạng mã hóa thống nhất, đồng thời chuyển về chữ thường để đánh giá tập trung vào mặt chữ thay vì in hoa/in thường.

Để đo lường hiệu suất của mô hình OCR, độ đo **CER (Character error rate - Tỉ lệ lỗi ký tự)** được sử dụng. CER dựa trên khoảng cách `Levenshtein`, đếm số lượng thao tác tối thiểu cần thiết để biến đổi chuỗi dự đoán thành chuỗi thực tế (ground truth).

Công thức tính toán:
$$CER = \frac{S + D + I}{N}$$

Trong đó:
* $S$ (Substitutions): Số ký tự bị thay thế sai.
* $D$ (Deletions): Số ký tự bị bỏ sót.
* $I$ (Insertions): Số ký tự bị chèn thừa.
* $N$: Tổng số ký tự của nhãn gốc (ground truth).

*Lưu ý:* Giá trị CER càng gần 0.0 thì mô hình nhận dạng càng chính xác.

In [ ]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFC', text)
    return text.strip().lower()

def calculate_cer(pred: str, gt: str) -> float:
    pred = normalize_text(pred)
    gt = normalize_text(gt)

    if len(gt) == 0:
        return 1.0 if len(pred) > 0 else 0.0

    edit_dist = Levenshtein.distance(pred, gt)
    cer = edit_dist / len(gt)
    return cer

### 2.1 Hàm thực thi mô hình GLMOCR
Thực hiện:
- Thay đổi kích thước về kích thước phù hợp với mô hình
- tạo cấu trúc hội thạo (message) theo định dạng mà mô hình yêu cầu, bao gồm `ảnh` và `prompt text`
- Sử dụng `processor` để chuẩn bị đầu vào cho mô hình
- Thực hiện suy luận (inference) không sử dụng gradient để tiết kiemeh bộ nhớ
- Giải mã kết quả đầu ra thành chuỗi văn bản dự đoán

In [ ]:
def _run_ocr_glm(model, processor, image, target_height=112, max_width=1120):
    orig_w, orig_h = image.size
    scale = target_height / orig_h
    new_w = max(28, (min(int(orig_w * scale), max_width) // 28) * 28)
    new_h = max(28, (int(orig_h * scale) // 28) * 28)
    image = image.resize((new_w, new_h), Image.LANCZOS)

    device = model.device

    # Cấu trúc hội thoại chuẩn của nhóm tác giả
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": "Text Recognition:"}
        ]
    }]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)

    generated = out[0][inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(generated, skip_special_tokens=True).strip()

## 3. Khởi tạo và đánh giá mô hình

Mô hình `zai-org/GLM-OCR` được tải trực tiếp từ Hugging Face. Vì đây là một kiến trúc mới, tham số `trust_remote_code=True` là bắt buộc để sử dụng các file mã nguồn tùy chỉnh của nhóm tác giả. 
Để mô hình có thể tải thành công trên GPU T4 của Google Colab mà không bị tràn RAM, trọng số được ép kiểu về dạng bán độ chính xác **FP16** (`torch_dtype=torch.float16`) và sử dụng cơ chế `device_map="auto"`.

**Quá trình đánh giá:**
- Thực thi hàm `_run_ocr_glm` như đã được mô tả ở bên trên
- Tính toán điểm CER cho từng ảnh, **chỉ lưu lại log của những ảnh có $CER > 0$** (những ảnh dự đoán sai). Việc này giúp tối ưu hóa bộ nhớ và tập trung hoàn toàn vào việc phân tích lỗi  ở bước sau.

In [5]:
# Khởi tạo mô hình
print("Đang nạp mô hình GLMOCR...")
BASE_MODEL = "zai-org/GLM-OCR"

cfg = AutoConfig.from_pretrained(BASE_MODEL, trust_remote_code=True)
module = importlib.import_module(f"transformers.models.{cfg.model_type.replace('-', '_')}")
gen_cls = next(
    (getattr(module, a) for a in dir(module) if "ForConditionalGeneration" in a),
    None
)

processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

model = gen_cls.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()

# Chuẩn bị danh sách ảnh
all_test_samples = []
subfolders = [f for f in TEST_DIR.iterdir() if f.is_dir()]
for subfolder in subfolders:
    label_file = subfolder / 'label.json'
    if not label_file.exists(): continue
    with open(label_file, 'r', encoding='utf-8') as f:
        ground_truths = json.load(f)
    for img_name, gt_text in ground_truths.items():
        img_path = subfolder / img_name
        if img_path.exists():
            all_test_samples.append((img_path, gt_text))

print(f"Thực hiện đánh giá trên {len(all_test_samples)} ảnh \n")

# Vòng lặp suy luận
total_cer = 0.0
error_logs = []

for img_path, gt_text in tqdm(all_test_samples, desc="Đang đánh giá"):
    try:
        img = Image.open(str(img_path))

        pred_text = _run_ocr_glm(model, processor, img)
    except Exception as e:
        pred_text = ""

    # Tính CER
    cer_score = calculate_cer(pred_text, gt_text)
    total_cer += cer_score

    if cer_score > 0:
        error_logs.append({
            "folder": img_path.parent.name,
            "image": img_path.name,
            "ground_truth": normalize_text(gt_text),
            "prediction": normalize_text(pred_text),
            "cer_score": round(cer_score, 4)
        })

# Kết quả tổng kết
test_samples = len(all_test_samples)
average_cer = total_cer / test_samples if test_samples > 0 else 0
print(f"\nCER trung bình: {average_cer * 100:.2f} %")
print(f"Số lượng ảnh dự đoán sai: {len(error_logs)}")

Đang nạp mô hình GLMOCR...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

Thực hiện đánh giá trên 15000 ảnh 



Đang đánh giá: 100%|██████████| 15000/15000 [1:20:39<00:00,  3.10it/s]


CER trung bình: 90.40 %
Số lượng ảnh dự đoán sai: 12599


## 4. Lưu trữ
Sau khi hoàn thành quá trình đánh giá, tạo ra file CSV `baseline_GLMOCR_report.csv` để báo cáo các lỗi và được lưu trên Google Drive, phục vụ cho quá trình phân tích, đối chiếu khi tinh chỉnh các mô hình sau này.

In [6]:
# Chuyển đổi danh sách lỗi thành Pandas DataFrame
df_errors = pd.DataFrame(error_logs)

# Sắp xếp từ lỗi nặng nhất (CER cao) xuống lỗi nhẹ nhất
df_errors = df_errors.sort_values(by="cer_score", ascending=False)

# Lưu file báo cáo ra ngoài thư mục Dataset_Splitted để dễ tìm
report_path = LOCAL_ROOT / 'baseline_PPOCR_report.csv'
df_errors.to_csv(report_path, index=False, encoding='utf-8-sig')

drive_report_path = Path('/content/drive/MyDrive/IntroToML - OCR - data/baseline_GLMOCR_report.csv')
!cp "{report_path}" "{drive_report_path}"

print(f"Đã lưu lại report của baseline_GLMOCR")

Đã lưu lại report của baseline_GLMOCR
